# V8 Electron Ptychography: Reconstruction Quick Start

**Supported methods:** Adam (default), ePIE and LSQML. This notebook uses existing YAML configurations and prepared `data/*.npz` datasets. It does not generate or modify input data.

**Environment requirements:** Python 3.11, Jupyter/IPython, NumPy, PyYAML, SciPy, h5py, Matplotlib, and a compatible GPU-enabled PyTorch installation. ePIE and LSQML additionally require Pty-Chi (`ptychi`). Step 01 checks the active Jupyter kernel and project files; Step 02 checks the Python environment used by the selected reconstruction method. No installation commands are executed.

**Project layout:** Place this notebook in `V08/` or `V08/notebooks/`, the YAML files in `V08/`, and the NPZ datasets in `V08/data/`. Dataset A/B/C/D presets below match the current project filenames.

**Path handling:** `run_platform.py` is always launched with the V08 project root as the working directory, so relative paths such as `data/...` and `outputs/...` are resolved from the project root. Other paths, including an optional separate Pty-Chi Python interpreter, may be absolute. Existing YAML files and datasets are not rewritten.

**Workflow:** 01. Check environment  |  02. Select and validate dataset and method  |  03. Run reconstruction.


## 01. Check the project and Python environment


In [2]:
from pathlib import Path
import importlib.util
import os
import sys

# Usually leave this empty. The notebook can be located in V08/ or V08/notebooks/.
# If automatic discovery fails, specify the project root, e.g. r"F:\Thesis\V08".
PROJECT_ROOT = ""

if PROJECT_ROOT.strip():
    root = Path(PROJECT_ROOT).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    root = next(
        (p for p in [here, *here.parents] if (p / "run_platform.py").is_file()),
        None,
    )
if root is None or not (root / "run_platform.py").is_file():
    raise FileNotFoundError("run_platform.py was not found. Set PROJECT_ROOT to the V08 project root.")

required_files = ["run_platform.py", "reconstruction.py", "forward_model.py"]
missing_files = [name for name in required_files if not (root / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing V08 project files: {missing_files}")

# Check packages in the selected notebook kernel without installing anything.
required_modules = {
    "numpy": "numpy", "PyYAML": "yaml", "scipy": "scipy",
    "h5py": "h5py", "matplotlib": "matplotlib", "torch": "torch",
    "IPython/Jupyter": "IPython",
}
missing_modules = [name for name, module in required_modules.items()
                   if importlib.util.find_spec(module) is None]
print("V08 root:", root)
print("Notebook Python:", sys.executable)
for name, module in required_modules.items():
    print(f"  {name:17s} {'MISSING' if name in missing_modules else 'OK'}")
if missing_modules:
    raise ModuleNotFoundError(f"Missing packages in the selected notebook kernel: {missing_modules}. Select a Jupyter kernel with the required packages.")

import numpy as np
import yaml
print("Environment and project file checks passed.")

V08 root: F:\Thesis\V08
Notebook Python: F:\Thesis\V08\venv\Scripts\python.exe
  numpy             OK
  PyYAML            OK
  scipy             OK
  h5py              OK
  matplotlib        OK
  torch             OK
  IPython/Jupyter   OK
Environment and project file checks passed.


## 02. Select and validate the dataset, YAML configuration and method


In [9]:
# Select the dataset and reconstruction method.
DATASET = "A"           # "A" / "B" / "C" / "D"; default: A
METHOD = "adam"         # "adam" / "epie" / "lsqml"; an empty value also selects Adam
# Set this to the separate Pty-Chi python.exe if needed. Otherwise, leave it empty.
PTYCHI_PYTHON = ""      # Example: r"F:\conda_envs\ptychi\python.exe"

# Presets match the current V08 YAML and NPZ filenames. Do not mix configurations and datasets.
PRESETS = {
    "A": ("config_abtem_a_v084_200pass_timed.yaml", "data/abtem_dataset_a_v8.npz", 1),
    "B": ("config_abtem_b_v084_200pass_timed.yaml", "data/abtem_dataset_b_pc_v8.npz", 2),
    "C": ("config_abtem_c_v084_200pass_timed.yaml", "data/abtem_dataset_c_pc_noise_v8.npz", 2),
    "D": ("config_abtem_graphene_complex_v084.yaml", "data/abtem_graphene_complex_v8.npz", 1),
}

DATASET = DATASET.strip().upper()
method = METHOD.strip().lower() or "adam"
if DATASET not in PRESETS:
    raise ValueError(f"DATASET must be one of {list(PRESETS)}. Dataset E has no predefined matching YAML.")
if method not in {"adam", "epie", "lsqml"}:
    raise ValueError("METHOD must be adam, epie or lsqml.")

yaml_name, expected_npz, expected_modes = PRESETS[DATASET]
config_path = root / yaml_name
prepared_path = root / expected_npz
for desc, path in [("YAML", config_path), ("Prepared NPZ", prepared_path)]:
    if not path.is_file():
        raise FileNotFoundError(f"{desc} was not found: {path}")

with config_path.open("r", encoding="utf-8-sig") as f:
    config = yaml.safe_load(f)
if not isinstance(config, dict):
    raise ValueError("The YAML root must be a mapping.")
for section in ("data", "forward", "reconstruction", "output"):
    if not isinstance(config.get(section), dict):
        raise ValueError(f"YAML is missing the {section} section.")

def resolve_v8_path(value):
    path = Path(str(value)).expanduser()
    return (path if path.is_absolute() else root / path).resolve()

yaml_npz = config["data"].get("prepared_npz_path")
if not yaml_npz:
    raise ValueError("The YAML does not define data.prepared_npz_path.")
if resolve_v8_path(yaml_npz) != prepared_path.resolve():
    raise ValueError(
        f"The YAML dataset path does not match Dataset {DATASET} preset:\n"
        f"YAML = {resolve_v8_path(yaml_npz)}\nPreset = {prepared_path}"
    )

recon = config["reconstruction"]
iterations = int(recon.get("iterations", 0))
modes = int(recon.get("n_modes", 0))
batch = int(recon.get("batch_size", 0))
if iterations <= 0 or batch <= 0 or modes != expected_modes:
    raise ValueError(
        f"The YAML reconstruction settings do not match the preset: iterations={iterations}, "
        f"batch_size={batch}, modes={modes} (expected modes={expected_modes})"
    )

output_value = config["output"].get("reconstruction_dir")
if not output_value:
    raise ValueError("The YAML does not define output.reconstruction_dir.")
output_base = resolve_v8_path(output_value)
result_dir = (output_base if method == "adam"
              else output_base.with_name(output_base.name + "_" + method))

print(f"Dataset {DATASET} | Method: {method.upper()}")
print("YAML:", config_path.relative_to(root))
print("Prepared NPZ:", prepared_path.relative_to(root))
print(f"Adam YAML settings: {iterations} iterations | batch {batch} | modes {modes}")
print("Output:", result_dir)

Dataset A | Method: ADAM
YAML: config_abtem_a_v084_200pass_timed.yaml
Prepared NPZ: data\abtem_dataset_a_v8.npz
Adam YAML settings: 200 iterations | batch 32 | modes 1
Output: F:\Thesis\V08\outputs\DatasetA_Coherent_1mode_V084_200pass_Timed


In [10]:
# Inspect NPZ array headers and limited metadata without loading all diffraction frames.
import json
import zipfile
from numpy.lib import format as npfmt

headers = {}
with zipfile.ZipFile(prepared_path) as zf:
    # Read array headers only; full zipfile.testzip() would decompress all diffraction frames.
    for member in zf.namelist():
        if not member.endswith(".npy"):
            continue
        with zf.open(member) as handle:
            version = npfmt.read_magic(handle)
            if version == (1, 0):
                shape, _fortran, dtype = npfmt.read_array_header_1_0(handle)
            elif version == (2, 0):
                shape, _fortran, dtype = npfmt.read_array_header_2_0(handle)
            elif version == (3, 0):
                # NumPy handles the UTF-8 NPY v3 header using its generic reader.
                shape, _fortran, dtype = npfmt._read_array_header(handle, version)
            else:
                raise ValueError(f"Unsupported NPY header version: {version}")
        headers[Path(member).stem] = (tuple(shape), np.dtype(dtype))

required = {"measured_intensity", "scan_positions", "metadata_json"}
if method == "adam":
    required |= {"crop_positions", "probe_pos_shifts"}
else:
    required |= {"ground_truth_object"}  # The current Pty-Chi benchmark runners still require ground truth.
missing = sorted(required - headers.keys())
if missing:
    raise ValueError(f"Missing NPZ fields required by the {method} runner: {missing}")

intensity_shape, intensity_dtype = headers["measured_intensity"]
if len(intensity_shape) != 3 or intensity_dtype.kind not in "fiu":
    raise ValueError(f"measured_intensity must be a real-valued array [N, H, W]; found {intensity_shape}, {intensity_dtype}")
nscan, det_y, det_x = intensity_shape
if headers["scan_positions"][0] != (nscan, 2):
    raise ValueError("scan_positions must have shape [N, 2], with coordinates ordered [y, x].")
if method == "adam":
    for name in ("crop_positions", "probe_pos_shifts"):
        if headers[name][0] != (nscan, 2):
            raise ValueError(f"{name} must have shape [N, 2].")
if tuple(config["forward"].get("detector_shape", [])) != (det_y, det_x):
    raise ValueError("YAML forward.detector_shape does not match the NPZ detector shape.")

with np.load(prepared_path, allow_pickle=False) as dataset:
    raw_meta = dataset["metadata_json"].item()
if isinstance(raw_meta, bytes):
    raw_meta = raw_meta.decode("utf-8")
metadata = json.loads(raw_meta) if isinstance(raw_meta, str) else raw_meta
if not isinstance(metadata, dict):
    raise ValueError("metadata_json must contain a valid JSON object.")
if method != "adam":
    if det_y != det_x:
        raise ValueError("The current Pty-Chi benchmark runner requires a square detector.")
    for key in ("probe_kv", "probe_conv_angle_mrad"):
        if key not in metadata:
            raise ValueError(f"The Pty-Chi runner requires metadata.{key}")
    runner_file = root / "engines" / "ptychi" / f"run_ptychi_{method}_v8.py"
    if not runner_file.is_file():
        raise FileNotFoundError(f"Missing reconstruction runner: {runner_file}")

print(f"NPZ validation passed: {nscan} patterns, detector {det_y} × {det_x}")
print("GT object:", "present" if "ground_truth_object" in headers else "absent")

# Check the Python interpreter used for reconstruction; it may differ from the notebook kernel.
import subprocess
runner_python = (Path(PTYCHI_PYTHON).expanduser().resolve()
                 if method != "adam" and PTYCHI_PYTHON.strip()
                 else Path(os.environ.get("PTYCHI_PYTHON", sys.executable)).expanduser().resolve()
                 if method != "adam" else Path(sys.executable).resolve())
if not runner_python.is_file():
    raise FileNotFoundError(f"Reconstruction Python interpreter was not found: {runner_python}")
check_code = (
    "import torch; "
    "print('PyTorch:', torch.__version__); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"
)
if method != "adam":
    check_code += "; import ptychi; print('Pty-Chi: OK')"
check = subprocess.run([str(runner_python), "-c", check_code],
                       cwd=str(root), capture_output=True, text=True)
print("Algorithm Python:", runner_python)
print(check.stdout.strip())
if check.returncode:
    raise RuntimeError("The selected reconstruction environment could not import torch/ptychi or initialize correctly. Check the kernel and PTYCHI_PYTHON.\n" + check.stderr[-2500:])
if "CUDA available: True" not in check.stdout:
    print("CUDA GPU was not detected. GPU reconstruction may fail or run substantially slower.")

NPZ validation passed: 3600 patterns, detector 124 × 124
GT object: present
Algorithm Python: F:\Thesis\V08\venv\Scripts\python.exe
PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5070


## 03. Run reconstruction

Standard output and error messages from the original reconstruction process are streamed into the notebook cell. The progress-bar presentation may differ from a terminal, but the underlying reconstruction is unchanged.


In [6]:
# -u and PYTHONUNBUFFERED enable prompt output from the reconstruction subprocess.
command = [sys.executable, '-u', str(root / 'run_platform.py'), 'reconstruct',
           '--config', str(config_path)]
if method != 'adam':
    command += ['--method', method]
    command += ['--ptychi-python', str(runner_python)]

print('Selected dataset:', prepared_path.relative_to(root))
print('Method:', method.upper())
print('YAML:', config_path.relative_to(root))
print('Expected output:', result_dir)
print('Command:', subprocess.list2cmdline(command))
if result_dir.is_dir() and any(result_dir.iterdir()):
    print('The output directory already contains files. Rerunning may overwrite files with the same names.')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
# Use UTF-8 output for the V8 process and its nested Pty-Chi runner.
env['PYTHONIOENCODING'] = 'utf-8:replace'
try:
    with subprocess.Popen(
        command,
        cwd=str(root),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # Stream log messages, progress updates and errors together.
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
    ) as process:
        # Forward each received line to the notebook output immediately.
        # Nested Pty-Chi runner output is forwarded through the V8 process.
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
except KeyboardInterrupt:
    # Terminate the direct subprocess if notebook execution is interrupted.
    # Check for any remaining child processes that may still use the GPU.
    if 'process' in locals() and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)
    raise
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
print('\nReconstruction finished. Output:', result_dir, flush=True)



Selected dataset: data\abtem_graphene_complex_v8.npz
Method: ADAM
YAML: config_abtem_graphene_complex_v084.yaml
Expected output: F:\Thesis\V08\outputs\DatasetD_Graphene_ComplexObject_V0840
Command: F:\Thesis\V08\venv\Scripts\python.exe -u F:\Thesis\V08\run_platform.py reconstruct --config F:\Thesis\V08\config_abtem_graphene_complex_v084.yaml
The output directory already contains files. Rerunning may overwrite files with the same names.

========== Reconstruction live output ==========

[V8] Method: Adam (default)
COMPUTE DEVICE
Requested: cuda
Selected: cuda:0
PyTorch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
Device: cuda:0
fitRBF: 25.115485 px
dx: 0.339665554 Å/object-pixel
scan step: 2.502461583 object pixels
initial position random std: 0.000000 px
object shape: (301, 297)
Object amplitude threshold: disabled (positive-only fallback active)
Probe power target: historical mean integrated measurement = 14710.043
Tensor devices:
  measured: cuda:0